# 01 从原文到抽取记录：真实结果回放

目标：辨认“原文、模型原始回答、结构化记录、课程参考”的区别，发现抽取漏掉的第二次实验。

本 Notebook 默认 **不联网、不调用 API、无需密钥**。使用 ReactionSeek 作者公开的真实 GLM 示例输出；回放不等于本次重新运行抽取模型。真实 DeepSeek 调用命令只放在文末 Markdown 中。

依次运行 01、02、03，三本共用 `runs/notebook-lab`。如需重做，请在启动 Jupyter 前把 `RKL_NOTEBOOK_RUN_DIR` 设置为新的目录，例如 `runs/notebook-lab-02`，三本使用相同设置。已有数据库不会被自动覆盖。

In [ ]:
from pathlib import Path
import os
import sys
import json

# 从仓库根目录或 notebooks/ 打开均可；不依赖个人机器的绝对路径。
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "pyproject.toml").is_file()
             and (p / "src/reaction_knowledge_lab").is_dir()), None)
if ROOT is None:
    raise RuntimeError("请在 reaction-knowledge-lab 仓库根目录或 notebooks 目录启动。")
sys.path.insert(0, str(ROOT / "src"))
run_setting = Path(os.environ.get("RKL_NOTEBOOK_RUN_DIR", "runs/notebook-lab")).expanduser()
LAB = run_setting.resolve() if run_setting.is_absolute() else (ROOT / run_setting).resolve()
LAB.mkdir(parents=True, exist_ok=True)
RAW, ALIGNED, DB = LAB / "extracted.csv", LAB / "aligned.json", LAB / "knowledge.sqlite"
from reaction_knowledge_lab.files import DATA, read_json, write_json, load_records, save_records
print("仓库：", ROOT)
print("本次实验目录：", LAB)
print("Python：", sys.executable)

## 1. 确认这批资料的身份

完整回放包有 5 篇文献、6 条抽取结果。必做实验只选 `volume102article22`，其他文献保留作扩展材料；不会把本例的名称缓存和实验编号规则直接套到其他论文。

In [ ]:
from reaction_knowledge_lab.extraction import CASE, replay
import csv
sources = read_json(DATA / "sources.json")
source = next(item for item in sources["sources"] if item["source_id"] == CASE)
print(json.dumps(source, ensure_ascii=False, indent=2))
with (DATA / "replay/upstream_output_table.csv").open(encoding="utf-8-sig", newline="") as stream:
    all_rows = list(csv.DictReader(stream))
print("上游回放行数：", len(all_rows))
print("主案例：", CASE)
assert len(all_rows) == 6

原文：*Photoexcited Nitroarenes for the Anaerobic Cleavage of Alkenes*，Organic Syntheses 102:114–127 (2025)。

[在线原文](https://www.orgsyn.org/demo.aspx?prep=v102p0114) · [PDF](https://www.orgsyn.org/Content/pdfs/procedures/v102p0114.pdf)

建议浏览 Procedure 的产物分离段及其中的重复实验描述。记录身份应对应一次实验，不应由产物名称或相同产率决定。

## 2. 回放作者的真实抽取结果

下面只复制这篇文献对应的既有结果，并写入零调用元数据。第一次运行应得到 **1 行**；若目录已有结果，请换新的实验目录，保留旧记录。

In [ ]:
if RAW.exists() or DB.exists():
    raise FileExistsError(f"已有实验结果：{LAB}。请使用新的 RKL_NOTEBOOK_RUN_DIR，不覆盖旧结果。")
metadata = replay(RAW, CASE)
print(json.dumps(metadata, ensure_ascii=False, indent=2))
assert metadata["mode"] == "replay"
assert metadata["api_calls"] == 0 and metadata["cost_cny"] == 0
assert metadata["row_count"] == 1

In [ ]:
with RAW.open(encoding="utf-8", newline="") as stream:
    extracted_rows = list(csv.DictReader(stream))
for field, value in extracted_rows[0].items():
    print(f"{field}: {value}")
assert extracted_rows[0]["Yield"] == "72%"
print("保存文件：", RAW)

注意 `1,1-diphenylethene` 的逗号属于化学名称。下一环节不能简单按逗号拆分全部名称，否则会制造错误实体。

## 3. 对照课程参考，找到漏抽的信息

`data/reference/case22.json` 是对照公开原文整理的**课程参考**，不是 ReactionSeek 作者提供的 gold，也未经独立化学专家审校。学生应利用其中的来源和证据位置复核它。

对照要点：原文两次实验均报告 72% 产率，但产物质量分别为 2.63 g 和 2.62 g，纯度也不同。既有抽取只留下第一条记录；第二次实验的条件没有在重复实验处逐项重述，后续应保留缺失，不能随意补全。

In [ ]:
reference = read_json(DATA / "reference/case22.json")
comparison = [{"record_id": r["record_id"], "mass_g": r["product"]["mass_g"],
               "yield_percent": r["product"]["yield_percent"],
               "purity_percent": r["product"]["purity_percent"]}
              for r in reference["records"]]
print(json.dumps(comparison, ensure_ascii=False, indent=2))
print("抽取条数 / 课程参考实验数：", len(extracted_rows), "/", len(comparison))
assert len(extracted_rows) == 1 and len(comparison) == 2
write_json(LAB / "extraction_comparison.json", {
    "mode": "upstream_replay", "extracted_count": len(extracted_rows),
    "reference_count": len(comparison), "missed_experiment": "orgsyn-102-0114-run-2",
    "reference_type": reference["reference_type"], "reference_summary": comparison,
})

## 4. 有原文时核对证据位置

完整原文不嵌进 Notebook。若已按照手册下载 `data/downloads/input.json`，下面会核对固定原文哈希与证据位置；未下载时会返回 `not_checked`，离线回放仍可继续，但不能声称已经重新核对了原文。只展示与质量/产率相关的短证据。

In [ ]:
from reaction_knowledge_lab.validation import check_evidence
print(json.dumps(check_evidence(), ensure_ascii=False, indent=2))
for item in reference["evidence"]:
    if item["field"] in {"product.mass_g", "product.yield_percent"}:
        print(item["record_id"], item["field"], f"[{item['start']}:{item['end']}]", item["text"])

## 5. 选做：真实调用路径（本 Notebook 不执行）

先按[主手册第 4 节](../docs/MANUAL.md#4-下载原始资料并检查证据)下载固定资料，再在真正的终端中执行[第 11.1 节](../docs/MANUAL.md#111-在终端安全输入密钥只调用一篇)的 `getpass` 命令。该命令隐藏密钥输入，只调用主案例一篇，使用 `--max-calls 1` 和 `--budget-cny 0.10`，输出到全新的 `runs/api/` 目录。

不要在 Notebook 单元格中粘贴密钥。若终端提示无法隐藏输入，停止输入并换终端。无需购买 API 的同学可以按手册第 11.2 节回放已保存的真实 DeepSeek 结果。

真实调用使用 DeepSeek 适配和原作者提示词，其输出可能与 GLM 回放不同。先保存和检查原始响应、usage、表格，再接后续阶段；不要为了得到固定答案而覆盖模型输出。本组 01–03 的自动验收以固定回放数据为基准。

**提交内容：** `extracted.csv`、`extracted.metadata.json`、`extraction_comparison.json`，以及你对“漏掉第二次实验为什么会影响知识库”的解释。接着打开 `02_alignment.ipynb`。